## Setup

In [2]:
from arcgis.gis import GIS
import arcpy
from arcgis.features import FeatureLayer
import pandas as pd
import json
import urllib.request
import urllib.parse
import os

In [3]:
gis = GIS('pro')
print(gis)
aprx = arcpy.mp.ArcGISProject("CURRENT")
map = aprx.activeMap
arcpy.env.addOutputsToMap = False

aprx.defaultGeodatabase = r"C:\Users\micha\OneDrive\Documents\ArcGIS\Projects\California Air Quality Sandbox\California Air Quality Sandbox.gdb"
arcpy.env.workspace = r"C:\Users\micha\OneDrive\Documents\ArcGIS\Projects\California Air Quality Sandbox\California Air Quality Sandbox.gdb"
reference_gdb = r"C:\Users\micha\OneDrive\Documents\ArcGIS\Projects\California Air Quality Sandbox\Data Sources.gdb"
working_gdb = r"C:\Users\micha\OneDrive\Documents\ArcGIS\Projects\California Air Quality Sandbox\California Air Quality Sandbox.gdb"

GIS @ https://www.arcgis.com/ version:[2026, 2]


In [4]:
print(aprx.defaultGeodatabase)
print(arcpy.env.workspace)
print(arcpy.ListFeatureClasses())

C:\Users\micha\OneDrive\Documents\ArcGIS\Projects\California Air Quality Sandbox\California Air Quality Sandbox.gdb
C:\Users\micha\OneDrive\Documents\ArcGIS\Projects\California Air Quality Sandbox\California Air Quality Sandbox.gdb
['EnviroScreen_CAPP_Summary', 'California_Air_Districts', 'EnviroScreen_CAPP_points']


#### Fixing EnviroScreen Geometry

In [ ]:
# Since I was seeing topographic errors in my geoprocessing, I went back and repaired whatever was here
arcpy.management.RepairGeometry(
   os.path.join(working_gdb, r"EnviroScreen_CAPP_Summary")
)
arcpy.management.CheckGeometry(
    os.path.join(working_gdb, r"EnviroScreen_CAPP_Summary"),
    os.path.join(working_gdb, r"geometry_errors_table")
)

In [ ]:
# Add a field to calculate geometry from scratch, just because
arcpy.management.AddField(os.path.join(working_gdb, r"EnviroScreen_CAPP_Summary"), "orig_tract_area", "DOUBLE")
arcpy.management.CalculateField(os.path.join(working_gdb, r"EnviroScreen_CAPP_Summary"), "orig_tract_area", "abs(!shape.area!)")

### Calculating CAPP Investment Reach

In [9]:
# Here we're creating a circle by iterating along a radius until we reach an investment's affected population
def find_reach_polygon(project_point, target_pop, pop_field="ACS2019TotalPop", coarse_step=500, fine_step=50, max_radius=50000):
    tract_path = os.path.join(working_gdb, "EnviroScreen_CAPP_Summary")

    # This sub-function will create a circle for the current iteration and add together all the partial populations from intersected tracts
    def get_total_pop(radius):
        arcpy.analysis.Buffer(project_point, "in_memory\\temp_buffer", f"{radius} Meters")
        arcpy.analysis.Intersect(["in_memory\\temp_buffer", tract_path], "in_memory\\temp_intersect")
        total_pop = 0
        with arcpy.da.SearchCursor("in_memory\\temp_intersect", [pop_field, "SHAPE@AREA", "orig_tract_area"]) as cursor:
            for pop, sliced_area, full_area in cursor:
                total_pop += pop * (sliced_area / full_area)
        return total_pop

    # I split this function into two parts for speed, since I really don't know how big any one circle will be
    # The first step is to iterate outward by a large step, until the circle exceeds the investment's affected population
    radius = coarse_step
    while radius <= max_radius:
        total_pop = get_total_pop(radius)
        if total_pop >= target_pop:
            break
        radius += coarse_step
    else:
        return None, total_pop  # never reached target within max_radius

    # The second step is to iterate inward by a small step, until the circle no longer exceeds the investment's affected population
    fine_radius = radius
    while fine_radius - fine_step > 0:
        test_pop = get_total_pop(fine_radius - fine_step)
        if test_pop < target_pop:
            break  # stepping back further would undershoot — stop here
        fine_radius -= fine_step
        total_pop = test_pop

    return fine_radius, total_pop

In [4]:
# Looping this over all CAPP incentives would have taken ~10 hours, so let's just look at the big ones.

with arcpy.da.SearchCursor(os.path.join(reference_gdb, "CAPP_Incentives"), ["grant_amount"]) as cursor:
    amounts = sorted([row[0] for row in cursor if row[0] is not None], reverse=True)

cutoff_value = amounts[int(len(amounts) * 0.01)]  # top 1% threshold
count_above_cutoff = sum(1 for a in amounts if a >= cutoff_value)
print(f"Number of projects above cutoff: {count_above_cutoff}")
print(f"cutoff: {cutoff_value}")

Number of projects above cutoff: 139
cutoff: 743000.0


In [ ]:
# Loop the circle-building function over the top 1% of CAPP investments

where_clause = f"grant_amount >= {cutoff_value}"

results = []
with arcpy.da.SearchCursor(os.path.join(reference_gdb, "CAPP_Incentives"), ["OID@", "SHAPE@", "grant_amount"], where_clause=where_clause) as cursor:
    rows = list(cursor)

print(f"Running on {len(rows)} large projects")

for i, (oid, shape, grant_amount) in enumerate(rows):
    print(f"[{i+1}/{len(rows)}] OID {oid}, ${grant_amount:,.0f}...")
    
    target_pop = grant_amount / 125  # Taken from CARB's statement of $500M affecting 4M Californians 

    # write this single point out so it can be used as buffer input
    single_pt = f"in_memory\\project_{oid}"
    arcpy.management.CopyFeatures(shape, single_pt)

    # reproject to match the tract layer's coordinate system
    single_pt_proj = f"in_memory\\project_proj_{oid}"
    arcpy.management.Project(single_pt, single_pt_proj, os.path.join(working_gdb, "EnviroScreen_CAPP_Summary"))

    radius, achieved_pop = find_reach_polygon(single_pt_proj, target_pop)

    if radius:
        final_name = os.path.join(working_gdb, f"reach_{oid}")
        arcpy.analysis.Buffer(single_pt_proj, final_name, f"{radius} Meters")
        results.append((oid, radius, achieved_pop))

    arcpy.management.Delete(single_pt)
    arcpy.management.Delete(single_pt_proj)

print(results)

In [ ]:
# Here we want to add the results to the existing CAPP incentives layer, passing the layer into a dataframe and back out into a new layer.

results_df = pd.DataFrame(results, columns=["OID", "radius", "achieved_pop"])

all_incentives_sedf = pd.DataFrame.spatial.from_featureclass(os.path.join(reference_gdb, "CAPP_Incentives"))

# left merge — keeps every row from all_incentives_sedf, fills NaN where no match in results_df
merged = all_incentives_sedf.merge(results_df, left_on="OBJECTID", right_on="OID", how="left")

merged.spatial.to_featureclass(location=os.path.join(working_gdb, "capp_reach_working"))